Objetivo: Descargar datos OHLCV reales de los 5 tickers del estudio usando la librería yfinance, calcular indicadores técnicos (SMA, EMA, RSI, MACD, Bollinger Bands) y exportar un JSON compatible con el archivo ernesto_investing_dashboard_mercado.html (adjunto como Anexo C.1).

# 1. Instalación de librerías: !pip install yfinance ta --quiet.

In [ ]:
# Instalación de librerías necesarias
!pip install yfinance ta --quiet

  Preparing metadata (setup.py) ... done


# 2. Importaciones: yfinance, pandas, numpy, ta, json

In [ ]:
# Importación de librerías
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import json
from datetime import datetime, timedelta

# 3. Configuración de tickers y periodo temporal

In [ ]:
# Configuración de los 5 activos financieros mineros con operaciones en Perú
TICKERS = {
    "FSM":        "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compañía Minera S.A.A.",
    "ABX.TO":     "Barrick Gold Corporation",
    "BVN":        "Compañía de Minas Buenaventura S.A.A.",
    "BHP":        "BHP Billiton Limited"
}

# Periodo temporal: últimos 2 años
FECHA_FIN   = datetime.today()
FECHA_INICIO = FECHA_FIN - timedelta(days=730)

FECHA_INICIO_STR = FECHA_INICIO.strftime("%Y-%m-%d")
FECHA_FIN_STR    = FECHA_FIN.strftime("%Y-%m-%d")

print(f"Periodo de descarga: {FECHA_INICIO_STR} → {FECHA_FIN_STR}")
print(f"Tickers configurados: {list(TICKERS.keys())}")

Periodo de descarga: 2024-06-16 → 2026-06-16
Tickers configurados: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']


# 4. Descarga de datos OHLCV con manejo de errores

In [ ]:
# Descarga de datos OHLCV reales desde Yahoo Finance
datos_raw = {}

for ticker, nombre in TICKERS.items():
    try:
        print(f"Descargando {ticker} ({nombre})...")
        df = yf.download(ticker, start=FECHA_INICIO_STR, end=FECHA_FIN_STR, progress=False)

        if df.empty:
            print(f"  ⚠️  Sin datos para {ticker}, se omite.")
            continue

        # Aplanar columnas MultiIndex si existen
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Renombrar columnas al estándar interno
        df = df.rename(columns={
            "Open":      "open",
            "High":      "high",
            "Low":       "low",
            "Close":     "close",
            "Adj Close": "adj_close",
            "Volume":    "volume"
        })

        df.index = pd.to_datetime(df.index)
        df = df.dropna()
        datos_raw[ticker] = df
        print(f"  ✅ {len(df)} registros descargados.")

    except Exception as e:
        print(f"  ❌ Error descargando {ticker}: {e}")

print(f"\nTickers descargados exitosamente: {list(datos_raw.keys())}")

Descargando FSM (Fortuna Silver Mines Inc.)...


/tmp/ipykernel_5356/3249766186.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=FECHA_INICIO_STR, end=FECHA_FIN_STR, progress=False)


  ✅ 500 registros descargados.
Descargando VOLCABC1.LM (Volcan Compañía Minera S.A.A.)...


/tmp/ipykernel_5356/3249766186.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=FECHA_INICIO_STR, end=FECHA_FIN_STR, progress=False)


  ✅ 493 registros descargados.
Descargando ABX.TO (Barrick Gold Corporation)...


...

  ✅ 501 registros descargados.
Descargando BVN (Compañía de Minas Buenaventura S.A.A.)...


/tmp/ipykernel_5356/3249766186.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=FECHA_INICIO_STR, end=FECHA_FIN_STR, progress=False)


  ✅ 500 registros descargados.
Descargando BHP (BHP Billiton Limited)...


/tmp/ipykernel_5356/3249766186.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=FECHA_INICIO_STR, end=FECHA_FIN_STR, progress=False)


  ✅ 500 registros descargados.

Tickers descargados exitosamente: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']


# 5. Cálculo de indicadores técnicos (SMA-20, SMA-50, EMA-12, EMA-26, RSI-14, MACD)

In [ ]:
# Cálculo de indicadores técnicos para cada ticker
datos_con_indicadores = {}

for ticker, df in datos_raw.items():
    print(f"Calculando indicadores para {ticker}...")
    d = df.copy()

    # --- Medias Móviles Simples ---
    d["sma_20"] = ta.trend.sma_indicator(d["close"], window=20)
    d["sma_50"] = ta.trend.sma_indicator(d["close"], window=50)

    # --- Medias Móviles Exponenciales ---
    d["ema_12"] = ta.trend.ema_indicator(d["close"], window=12)
    d["ema_26"] = ta.trend.ema_indicator(d["close"], window=26)

    # --- RSI (Relative Strength Index) ---
    d["rsi_14"] = ta.momentum.rsi(d["close"], window=14)

    # --- MACD ---
    macd_obj     = ta.trend.MACD(d["close"], window_slow=26, window_fast=12, window_sign=9)
    d["macd"]         = macd_obj.macd()
    d["macd_signal"]  = macd_obj.macd_signal()
    d["macd_hist"]    = macd_obj.macd_diff()

    # --- Bollinger Bands ---
    bb_obj            = ta.volatility.BollingerBands(d["close"], window=20, window_dev=2)
    d["bb_upper"]     = bb_obj.bollinger_hband()
    d["bb_middle"]    = bb_obj.bollinger_mavg()
    d["bb_lower"]     = bb_obj.bollinger_lband()

    # --- Señal BUY/SELL basada en cruce SMA-20 y SMA-50 ---
    d["signal"] = "HOLD"
    d.loc[d["sma_20"] > d["sma_50"], "signal"] = "BUY"
    d.loc[d["sma_20"] < d["sma_50"], "signal"] = "SELL"

    # Eliminar filas con NaN generados por los indicadores
    d = d.dropna()
    datos_con_indicadores[ticker] = d
    print(f"  ✅ Indicadores calculados. Registros finales: {len(d)}")

Calculando indicadores para FSM...
  ✅ Indicadores calculados. Registros finales: 451
Calculando indicadores para VOLCABC1.LM...
  ✅ Indicadores calculados. Registros finales: 444
Calculando indicadores para ABX.TO...
  ✅ Indicadores calculados. Registros finales: 452
Calculando indicadores para BVN...
  ✅ Indicadores calculados. Registros finales: 451
Calculando indicadores para BHP...
  ✅ Indicadores calculados. Registros finales: 451


# 6. Transformación a estructura JSON del frontend

In [ ]:
# Transformación de los datos al formato JSON que consume el frontend
def redondear(valor):
    """Redondea un valor numérico a 4 decimales, maneja NaN."""
    try:
        if pd.isna(valor):
            return None
        return round(float(valor), 4)
    except:
        return None

output = {
    "metadata": {
        "generado_en":   datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "periodo_inicio": FECHA_INICIO_STR,
        "periodo_fin":    FECHA_FIN_STR,
        "fuente":         "Yahoo Finance (yfinance)",
        "tickers":        list(datos_con_indicadores.keys())
    },
    "activos": {}
}

for ticker, df in datos_con_indicadores.items():
    nombre = TICKERS.get(ticker, ticker)

    # Precio y variación más recientes
    precio_actual  = redondear(df["close"].iloc[-1])
    precio_anterior = redondear(df["close"].iloc[-2])
    variacion_dia  = redondear(precio_actual - precio_anterior) if precio_actual and precio_anterior else None
    variacion_pct  = redondear((variacion_dia / precio_anterior) * 100) if precio_anterior else None

    # Serie histórica diaria
    serie = []
    for fecha, fila in df.iterrows():
        serie.append({
            "fecha":       fecha.strftime("%Y-%m-%d"),
            "open":        redondear(fila["open"]),
            "high":        redondear(fila["high"]),
            "low":         redondear(fila["low"]),
            "close":       redondear(fila["close"]),
            "adj_close":   redondear(fila.get("adj_close")),
            "volume":      int(fila["volume"]) if not pd.isna(fila["volume"]) else None,
            "sma_20":      redondear(fila["sma_20"]),
            "sma_50":      redondear(fila["sma_50"]),
            "ema_12":      redondear(fila["ema_12"]),
            "ema_26":      redondear(fila["ema_26"]),
            "rsi_14":      redondear(fila["rsi_14"]),
            "macd":        redondear(fila["macd"]),
            "macd_signal": redondear(fila["macd_signal"]),
            "macd_hist":   redondear(fila["macd_hist"]),
            "bb_upper":    redondear(fila["bb_upper"]),
            "bb_middle":   redondear(fila["bb_middle"]),
            "bb_lower":    redondear(fila["bb_lower"]),
            "signal":      fila["signal"]
        })

    output["activos"][ticker] = {
        "nombre":          nombre,
        "ticker":          ticker,
        "precio_actual":   precio_actual,
        "variacion_dia":   variacion_dia,
        "variacion_pct":   variacion_pct,
        "signal_actual":   df["signal"].iloc[-1],
        "rsi_actual":      redondear(df["rsi_14"].iloc[-1]),
        "macd_actual":     redondear(df["macd"].iloc[-1]),
        "volumen_actual":  int(df["volume"].iloc[-1]),
        "total_registros": len(df),
        "serie":           serie
    }

print("✅ Estructura JSON construida correctamente.")
for ticker, datos in output["activos"].items():
    print(f"   {ticker}: {datos['total_registros']} registros | Signal: {datos['signal_actual']} | Precio: {datos['precio_actual']}")

✅ Estructura JSON construida correctamente.
   FSM: 451 registros | Signal: SELL | Precio: 9.45
   VOLCABC1.LM: 444 registros | Signal: BUY | Precio: 0.88
   ABX.TO: 452 registros | Signal: SELL | Precio: 58.62
   BVN: 451 registros | Signal: SELL | Precio: 34.86
   BHP: 451 registros | Signal: BUY | Precio: 92.13


# 7. Exportación a archivo datos_mercado.json.

In [ ]:
# Exportación del JSON final al archivo de contrato de datos
NOMBRE_ARCHIVO = "datos_mercado.json"

with open(NOMBRE_ARCHIVO, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

# Verificación del archivo generado
import os
tamanio_kb = os.path.getsize(NOMBRE_ARCHIVO) / 1024
print(f"✅ Archivo exportado: {NOMBRE_ARCHIVO}")
print(f"   Tamaño: {tamanio_kb:.1f} KB")
print(f"   Tickers incluidos: {list(output['activos'].keys())}")
print(f"   Generado en: {output['metadata']['generado_en']}")

# Vista previa de las primeras líneas del JSON
with open(NOMBRE_ARCHIVO, "r", encoding="utf-8") as f:
    preview = f.read(800)
print(f"\n--- Vista previa del JSON ---\n{preview}...")

✅ Archivo exportado: datos_mercado.json
   Tamaño: 1248.0 KB
   Tickers incluidos: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Generado en: 2026-06-16 21:23:59

--- Vista previa del JSON ---
{
  "metadata": {
    "generado_en": "2026-06-16 21:23:59",
    "periodo_inicio": "2024-06-16",
    "periodo_fin": "2026-06-16",
    "fuente": "Yahoo Finance (yfinance)",
    "tickers": [
      "FSM",
      "VOLCABC1.LM",
      "ABX.TO",
      "BVN",
      "BHP"
    ]
  },
  "activos": {
    "FSM": {
      "nombre": "Fortuna Silver Mines Inc.",
      "ticker": "FSM",
      "precio_actual": 9.45,
      "variacion_dia": 0.52,
      "variacion_pct": 5.8231,
      "signal_actual": "SELL",
      "rsi_actual": 50.6377,
      "macd_actual": -0.2993,
      "volumen_actual": 5446000,
      "total_registros": 451,
      "serie": [
        {
          "fecha": "2024-08-27",
          "open": 4.77,
          "high": 4.8,
          "low": 4.7,
          "close": 4.78,
          "adj_close": null,
     ...